In [1]:
import time
import os
import re

import porespy as ps
import numpy as np
import scipy as sc
import xarray as xr
from pypardiso import spsolve
from scipy.sparse import csr_matrix

os.chdir("..")
%run .\pyflowsolver\volumeManager.py
%run .\pyflowsolver\sparseArray.py
%run .\pyflowsolver\fastLaplacian.py
%run .\pyflowsolver\darcySolver.py
os.chdir("notebooks")

In [2]:
def solve_array(vol, scale=(1., 1., 1.), solver_type="pypardiso", target_error=1e-5):
    
    size = vol.shape
    scaled_size = [(a*b) for (a,b) in zip(scale, size)]

    vol, n_lab = sc.ndimage.label(vol)
    vol = (vol==1)
    
    cond_vol = vol*100 #porosity map ndarray uint8 0..100
    cond_vol = fast_laplacian_volume_generator(
        cond_vol, 
        scale, 
        )
    volume_manager = VolumeManager(cond_vol)
    
    solver = DarcySolver()
    sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

    start_time = time.perf_counter()
    if solver_type == "pypardiso":
        A = csr_matrix( (sparse_A["val"], sparse_A["col_idx"], np.append(sparse_A["row_ptr"],sparse_A["val"].size)) )
        solution = spsolve(A, sparse_b)
        error = 0
        iterations = 0
    elif solver_type == "pyflowsolver":
        max_iterations = sparse_b.size
        P_val, P_col_idx, P_row_ptr = _get_diagonal_preconditioner(
        A_val = sparse_A["val"], 
        A_col_idx=sparse_A["col_idx"], 
        A_row_ptr=sparse_A["row_ptr"], 
        threads=1,
        )
        solution, error, iterations = solver._solve_pcg(
            sparse_A["val"],
            sparse_A["col_idx"],
            sparse_A["row_ptr"],
            P_val, 
            P_col_idx, 
            P_row_ptr,
            sparse_b,
            max_iterations=max_iterations*100, # sqrt(n) for n x n system
            target_error=target_error, # 1.0e-6
            X0=np.zeros(sparse_b.size, dtype=np.float64),
            threads=1,
        )
        
    pressure = volume_manager.ravel_sparse_solution(solution)
    run_time = time.perf_counter() - start_time

    z_gradient = np.zeros_like(pressure)
    z_gradient[:,:,:-1] = pressure[:,:,:-1] - pressure[:,:,1:]
    z_gradient[:,:,-1] = pressure[:,:,-1]
    z_cond = np.zeros_like(pressure)
    z_cond[:,:,:-1] = 2 / (1/cond_vol[:,:,:-1] + 1/cond_vol[:,:,1:])
    z_cond[:,:,-1] = 2 * cond_vol[:,:,-1]
    z_speed = z_gradient * z_cond
    q_z = (z_speed[:,:,0].sum() + z_speed[:,:,-1].sum())/2
    k_z = (q_z * scaled_size[2]) / (scaled_size[0] * scaled_size[1])
    
    return pressure, z_speed, k_z, error, iterations, run_time


In [3]:
folder = r"D:\Geoslicer\benthimer_eval\numerical_compare\netcdf"
files = {}
for f in os.listdir(folder):
    if f[-3:] != ".nc":
        continue
    match = re.search(r'([02]{3})\.nc$', f)
    if match:
        files[match[1]] = f
        continue
    match = re.search(r'([13]{3})\.nc$', f)
    if match:
        files[match[1]] = f
        continue


In [4]:
SOLVER = ("pypardiso","pyflowsolver")[1]
LENGTH=250
for key, file in files.items():
    #if file != "BIN_Bentheimer313.nc": continue
    dataset = xr.open_dataset(f"{folder}\{file}")
    numpy_array = dataset['__xarray_dataarray_variable__'].values
    numpy_array = (1 - numpy_array)
    vol = np.swapaxes(numpy_array, 0, 2)
    mirror_vol = np.flip(vol, axis=1)
    vol = np.concatenate((vol, mirror_vol), axis=1)
    vol, n_lab = sc.ndimage.label(vol)
    intersect = np.intersect1d(np.unique(vol[:,:,0]), np.unique(vol[:,:,-1]))
    intersect = intersect[intersect != 0]
    vol = np.isin(vol, intersect).astype(np.uint8)
    pressure, speed, k_z, error, iterations, run_time = solve_array(
        #vol[:LENGTH, :LENGTH, :LENGTH], 
        vol,
        solver_type=SOLVER,
        target_error=1e-6
    )
    print(f"{file}\tPermeability: {k_z:.4f}\tRun time: {run_time:.4f}\tError: {error:.4f}\tIter: {iterations}")
    coords= {
        'x':np.arange(LENGTH),
        'y':np.arange(LENGTH),
        'z':np.arange(LENGTH),
    }
    #pressure_dataarray = xr.DataArray(pressure, coords)
    #speed_dataarray = xr.DataArray(speed, coords)
    #dataset = xr.Dataset({"pressure": pressure_dataarray, "speed": speed_dataarray})
    #dataset.to_netcdf(f"{folder}\solved\{file}")

    

BIN_Bentheimer000.nc	Permeability: 0.6167	Run time: 608.8865	Error: 0.0000	Iter: 2480
BIN_Bentheimer002.nc	Permeability: 0.9856	Run time: 634.2843	Error: 0.0000	Iter: 2833
BIN_Bentheimer020.nc	Permeability: 7.5589	Run time: 586.9624	Error: 0.0000	Iter: 1901
BIN_Bentheimer022.nc	Permeability: 0.6059	Run time: 499.4475	Error: 0.0000	Iter: 2182
BIN_Bentheimer111.nc	Permeability: 0.8675	Run time: 511.9055	Error: 0.0000	Iter: 2137
BIN_Bentheimer113.nc	Permeability: 0.7017	Run time: 573.5956	Error: 0.0000	Iter: 2560
BIN_Bentheimer131.nc	Permeability: 0.9047	Run time: 517.0626	Error: 0.0000	Iter: 2125
BIN_Bentheimer133.nc	Permeability: 0.8846	Run time: 619.7873	Error: 0.0000	Iter: 2380
BIN_Bentheimer200.nc	Permeability: 1.1339	Run time: 527.1573	Error: 0.0000	Iter: 2099
BIN_Bentheimer202.nc	Permeability: 0.6037	Run time: 690.5162	Error: 0.0000	Iter: 3420
BIN_Bentheimer220.nc	Permeability: 7.1397	Run time: 889.1446	Error: 0.0000	Iter: 2631
BIN_Bentheimer222.nc	Permeability: 1.1533	Run time: 46

In [5]:
dataset = xr.open_dataset(f"{folder}\BIN_Bentheimer313.nc")

In [6]:
numpy_array = dataset['__xarray_dataarray_variable__'].values

In [7]:
numpy_array = (1 - numpy_array)

In [8]:
vol = np.swapaxes(numpy_array, 0, 2)

In [9]:
vol, n_lab = sc.ndimage.label(vol)

In [10]:
np.unique(vol[:,:,0])

array([ 0,  1,  2,  7, 54, 58])

In [11]:
np.unique(vol[:,:,-1])

array([ 0,  2, 60])

In [12]:
set(np.unique(vol[:,:,0])), set(np.unique(vol[:,:,-1]))

({0, 1, 2, 7, 54, 58}, {0, 2, 60})

In [13]:
intersect = np.intersect1d(np.unique(vol[:,:,0]), np.unique(vol[:,:,-1]))

In [14]:
intersect[intersect != 0]

array([2])